# Fine-tuning Transformer Models for Single-Label Sentiment Classification

In this notebook, transformer-based language models are fine-tuned for sentiment analysis of Shopify customer reviews. The aim is to classify each review into one sentiment category: negative, neutral, or positive.

The models used in this study are BERT, RoBERTa, and DistilBERT. The same dataset and similar training structure are used for all models in order to make the comparison fair.

The original notebook structure was adapted for this thesis. The code was modified to load a custom Shopify review dataset, clean the review texts, translate non-English reviews into English, convert 1–5 star ratings into sentiment labels, train the selected transformer models, and evaluate their classification performance.

The models are compared using accuracy, precision, recall, F1-score, confusion matrix, prediction distribution, inference time, and GPU memory usage.

In [ ]:
!uv pip install transformers datasets scikit-learn torch --quiet

In [ ]:
import random
import numpy as np
import torch
from transformers import set_seed

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [ ]:
import pandas as pd

df = pd.read_excel("shopify_reviews.xlsx")
df.head()

,content,email,product_id,status,name,photo_urls,created_at,rating,product_title,product_sku,product_url,product_handle
0,i used to have a feliway - they can be interch...,NaN,15735008788853,published,Halina Król,NaN,2025-08-17T00:00:00.000Z,5,Cats Calming Diffuser Kit,NaN,https://us1cxf-vd.myshopify.com/products/cats-...,cats-calming-diffuser-kit
1,NaN,NaN,15735008788853,published,Daniel Castro,NaN,2026-02-15T00:00:00.000Z,5,Cats Calming Diffuser Kit,NaN,https://us1cxf-vd.myshopify.com/products/cats-...,cats-calming-diffuser-kit
2,NaN,NaN,15735008788853,published,Mariana Pérez,NaN,2025-12-05T00:00:00.000Z,5,Cats Calming Diffuser Kit,NaN,https://us1cxf-vd.myshopify.com/products/cats-...,cats-calming-diffuser-kit
3,"Excelente producto, los gatos se han calmado e...",NaN,15735008788853,published,Sofía Silva,NaN,2025-12-29T00:00:00.000Z,5,Cats Calming Diffuser Kit,NaN,https://us1cxf-vd.myshopify.com/products/cats-...,cats-calming-diffuser-kit
4,NaN,NaN,15707298595189,published,Noa Cohen,NaN,2026-02-20T00:00:00.000Z,5,PetPacify Dog Calming Diffuser,NaN,https://us1cxf-vd.myshopify.com/products/petpa...,petpacify-dog-calming-diffuser-kit


In [ ]:

from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict


df = df[["content", "rating"]].copy()

def clean_text(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    x = " ".join(x.split())
    if len(x) == 0:
        return None
    return x

def map_label(rating):
    try:
        r = float(rating)
    except:
        return None

    if r in [1.0, 2.0]:
        return 0   # negative
    elif r == 3.0:
        return 1   # neutral
    elif r in [4.0, 5.0]:
        return 2   # positive
    return None

df["content"] = df["content"].apply(clean_text)
df["label"] = df["rating"].apply(map_label)

df = df.dropna(subset=["content", "label"])
df = df.drop_duplicates(subset=["content", "label"]).reset_index(drop=True)

df["label"] = df["label"].astype(int)

label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {0: "negative", 1: "neutral", 2: "positive"}
labels = ["negative", "neutral", "positive"]

print(df["label"].value_counts())

train_df, temp_df = train_test_split(
    df,
    test_size=0.4,
    random_state=42,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label"]
)

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "validation": Dataset.from_pandas(val_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True)),
})

dataset

label
2    255
0     25
1     16
Name: count, dtype: int64


DatasetDict({
    train: Dataset({
        features: ['content', 'rating', 'label'],
        num_rows: 177
    })
    validation: Dataset({
        features: ['content', 'rating', 'label'],
        num_rows: 59
    })
    test: Dataset({
        features: ['content', 'rating', 'label'],
        num_rows: 60
    })
})

In [ ]:
labels = ["negative", "neutral", "positive"]

id2label = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

labels

['negative', 'neutral', 'positive']

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "bert-base-uncased"

# Later, change only this line:
# model_checkpoint = "bert-base-uncased"
# model_checkpoint = "roberta-base"

output_folder = f"./results_{model_checkpoint.replace('/', '_')}"
logging_folder = f"./logs_{model_checkpoint.replace('/', '_')}"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_data(examples):
    encoding = tokenizer(
        examples["content"],
        padding="max_length",
        truncation=True,
        max_length=128
    )
    encoding["labels"] = examples["label"]
    return encoding

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
encoded_dataset = dataset.map(
    preprocess_data,
    batched=True,
    remove_columns=dataset["train"].column_names
)

encoded_dataset.set_format("torch")

Map:   0%|          | 0/177 [00:00<?, ? examples/s]

Map:   0%|          | 0/59 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

In [ ]:
example = encoded_dataset["train"][0]

print(example.keys())
print(tokenizer.decode(example["input_ids"]))
print(example["labels"])
print(id2label[int(example["labels"])])

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels'])
[CLS] works well [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
tensor(2)
positive


In [ ]:
example = encoded_dataset["train"][0]

print(example["labels"])
print(id2label[int(example["labels"])])

tensor(2)
positive


In [ ]:
print(example["labels"])
print(id2label[int(example["labels"])])

tensor(2)
positive


In [ ]:
encoded_dataset.set_format("torch")

In [ ]:
from transformers import AutoModelForSequenceClassification
import torch

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

print("Checkpoint:", model_checkpoint)
print("Model name:", model.config._name_or_path)
print("Number of labels:", model.config.num_labels)
print("Parameter count:", sum(p.numel() for p in model.parameters()))

if torch.cuda.is_available():
    model.to("cuda")
    print("Device: CUDA")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Device: CPU")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Checkpoint: bert-base-uncased
Model name: bert-base-uncased
Number of labels: 3
Parameter count: 109484547
Device: CUDA
GPU: Tesla T4


In [ ]:
batch = {k: v.unsqueeze(0).to(model.device) for k, v in encoded_dataset["train"][0].items()}

with torch.no_grad():
    outputs = model(**batch)

print(outputs.logits)
print(outputs.logits.shape)

tensor([[ 0.0361,  0.0302, -0.0678]], device='cuda:0')
torch.Size([1, 3])


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import EvalPrediction

def compute_metrics(eval_pred: EvalPrediction):
    logits, labels_true = eval_pred
    preds = np.argmax(logits, axis=1)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels_true,
        preds,
        average="macro",
        zero_division=0
    )

    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels_true,
        preds,
        average="weighted",
        zero_division=0
    )

    accuracy = accuracy_score(labels_true, preds)

    return {
        "accuracy": accuracy,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted
    }

In [ ]:
from transformers import TrainingArguments, Trainer

batch_size = 8
num_train_epochs = 8
learning_rate = 2e-5

args = TrainingArguments(
    output_dir=output_folder,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=learning_rate,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_train_epochs,
    weight_decay=0.01,
    logging_dir=logging_folder,
    load_best_model_at_end=False,
    report_to="none",
    seed=42,
    data_seed=42,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
encoded_dataset['train'][0]['labels'].type()

'torch.LongTensor'

In [ ]:
encoded_dataset['train']['input_ids'][0]

tensor([ 101, 2573, 2092,  102,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0])

In [ ]:
batch = {
    k: v.unsqueeze(0).to(model.device)
    for k, v in encoded_dataset["train"][0].items()
}

with torch.no_grad():
    outputs = model(**batch)

print(outputs.loss)
print(outputs.logits)
print(outputs.logits.shape)

tensor(1.1670, device='cuda:0')
tensor([[ 0.0361,  0.0302, -0.0678]], device='cuda:0')
torch.Size([1, 3])


In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
1,No log,0.522510,0.864407,0.288136,0.333333,0.309091,0.747199,0.864407,0.801541
2,No log,0.426823,0.864407,0.288136,0.333333,0.309091,0.747199,0.864407,0.801541
3,No log,0.393919,0.864407,0.288136,0.333333,0.309091,0.747199,0.864407,0.801541
4,No log,0.385912,0.864407,0.288136,0.333333,0.309091,0.747199,0.864407,0.801541
5,No log,0.384690,0.864407,0.288136,0.333333,0.309091,0.747199,0.864407,0.801541
6,No log,0.379282,0.881356,0.626437,0.400000,0.423038,0.844828,0.881356,0.837143
7,No log,0.408339,0.881356,0.626437,0.400000,0.423038,0.844828,0.881356,0.837143
8,No log,0.462180,0.881356,0.626437,0.400000,0.423038,0.844828,0.881356,0.837143


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=184, training_loss=0.3875094289365022, metrics={'train_runtime': 462.5449, 'train_samples_per_second': 3.061, 'train_steps_per_second': 0.398, 'total_flos': 93142149875712.0, 'train_loss': 0.3875094289365022, 'epoch': 8.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 0.46217992901802063,
 'eval_accuracy': 0.8813559322033898,
 'eval_precision_macro': 0.6264367816091955,
 'eval_recall_macro': 0.39999999999999997,
 'eval_f1_macro': 0.4230377166156983,
 'eval_precision_weighted': 0.8448275862068966,
 'eval_recall_weighted': 0.8813559322033898,
 'eval_f1_weighted': 0.8371430052350594,
 'eval_runtime': 0.4792,
 'eval_samples_per_second': 123.11,
 'eval_steps_per_second': 16.693,
 'epoch': 8.0}

In [ ]:
import time
import psutil
import os
import torch

# Make sure the model is in evaluation mode
trainer.model.eval()

# Clear previous GPU memory statistics
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

# Measure RAM before prediction
process = psutil.Process(os.getpid())
ram_before = process.memory_info().rss / (1024 ** 2)  # MB

# Measure inference time
start_time = time.time()

pred_output = trainer.predict(encoded_dataset["test"])

if torch.cuda.is_available():
    torch.cuda.synchronize()

end_time = time.time()

# Measure RAM after prediction
ram_after = process.memory_info().rss / (1024 ** 2)  # MB

# Calculate results
total_inference_time = end_time - start_time
average_inference_time = total_inference_time / len(encoded_dataset["test"])
ram_usage = ram_after - ram_before

if torch.cuda.is_available():
    gpu_peak_memory = torch.cuda.max_memory_allocated() / (1024 ** 2)  # MB
else:
    gpu_peak_memory = None

print("Model:", model_checkpoint)
print("Total inference time (seconds):", round(total_inference_time, 4))
print("Average inference time per sample (seconds):", round(average_inference_time, 6))
print("RAM usage during prediction (MB):", round(ram_usage, 2))
print("GPU peak memory during prediction (MB):", round(gpu_peak_memory, 2) if gpu_peak_memory is not None else "No GPU")

Model: bert-base-uncased
Total inference time (seconds): 0.4023
Average inference time per sample (seconds): 0.006705
RAM usage during prediction (MB): 0.0
GPU peak memory during prediction (MB): 1574.91


In [ ]:
text = "I'm happy I can finally train a model for single-label classification"

encoding = tokenizer(text, return_tensors="pt")
encoding = {k: v.to(trainer.model.device) for k,v in encoding.items()}

outputs = trainer.model(**encoding)

In [ ]:
logits = outputs.logits
logits.shape

torch.Size([1, 3])

In [ ]:
# apply sigmoid + threshold
sigmoid = torch.nn.Sigmoid()
probs = sigmoid(logits.squeeze().cpu())
predictions = np.zeros(probs.shape)
predictions[np.where(probs >= 0.5)] = 1
# turn predicted id's into actual label names
predicted_labels = [id2label[idx] for idx, label in enumerate(predictions) if label == 1.0]
print(predicted_labels)

['positive']


In [ ]:
test_results = trainer.evaluate(encoded_dataset["test"])
test_results

{'eval_loss': 0.48575884103775024,
 'eval_accuracy': 0.8666666666666667,
 'eval_precision_macro': 0.2888888888888889,
 'eval_recall_macro': 0.3333333333333333,
 'eval_f1_macro': 0.30952380952380953,
 'eval_precision_weighted': 0.7511111111111112,
 'eval_recall_weighted': 0.8666666666666667,
 'eval_f1_weighted': 0.8047619047619048,
 'eval_runtime': 0.4155,
 'eval_samples_per_second': 144.395,
 'eval_steps_per_second': 19.253,
 'epoch': 8.0}

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

pred_output = trainer.predict(encoded_dataset["test"])

preds = np.argmax(pred_output.predictions, axis=1)
true_labels = pred_output.label_ids

print("True label distribution:")
print(pd.Series(true_labels).map(id2label).value_counts())

print("\nPredicted label distribution:")
print(pd.Series(preds).map(id2label).value_counts())

print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, preds))

print("\nClassification Report:")
print(classification_report(true_labels, preds, target_names=labels, zero_division=0))

True label distribution:
positive    52
negative     5
neutral      3
Name: count, dtype: int64

Predicted label distribution:
positive    60
Name: count, dtype: int64

Confusion Matrix:
[[ 0  0  5]
 [ 0  0  3]
 [ 0  0 52]]

Classification Report:
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00         5
     neutral       0.00      0.00      0.00         3
    positive       0.87      1.00      0.93        52

    accuracy                           0.87        60
   macro avg       0.29      0.33      0.31        60
weighted avg       0.75      0.87      0.80        60

